In [1]:
import pandas as pd
import numpy as np
import yaml
from collections import defaultdict

## Load data

In [2]:
path_to_file = "data/mmc.yaml"

with open(path_to_file, "r", encoding="utf-8") as file:
    raw_data = yaml.safe_load(file)
    
df = pd.DataFrame(raw_data)

df_naselja = pd.read_csv("./processed_data/naselja.csv")

df_regije = df_naselja[['region_id', 'region_name']].drop_duplicates().values

In [3]:
df = df.sample(frac=0.1) #Keep only 10% of all data for testing

In [4]:
df = df[df['paragraphs'].str.len() > 0]
empty_values = ["", "[]"]
df = df[~df['topics'].isin(empty_values)]

In [5]:
# Clean text column, kjer je vse lower case in nima stopwordov

with open('./data/stopwords-sl.txt', 'r', encoding='utf-8') as f:
    sl_stopwords = [line.strip() for line in f if line.strip()]

def clean_data(text_lst):
    full_text = " ".join(text_lst)
    clean_text = "".join(c.lower() if c.isalpha() or c in "čšžćđ-" else " " for c in full_text)
    padded_text = f" {clean_text} "    
    words = padded_text.split()
    final_words = [w for w in words if w not in sl_stopwords]
    return " ".join(final_words)

df["clean_text"] = df["paragraphs"].apply(clean_data)
df["paragraphs"] = df["paragraphs"].apply(
    lambda ps: "\n\n".join(ps) if isinstance(ps, list) else str(ps)
)

In [6]:
# Odstrani naselja, ki imajo več kot 1 lokacijo 

region_counts = df_naselja.groupby('naselje')['region_name'].transform('nunique')
ambiguous_mask = region_counts > 1
df_removed = df_naselja[ambiguous_mask].sort_values(by='naselje')
df_naselja = df_naselja[~ambiguous_mask].copy()
df_naselja = df_naselja.drop_duplicates(subset=['naselje', 'region_name'])

In [ ]:
df.head(3)

,_id,url,authors,date,title,paragraphs,figures,lead,mention,topics,keywords,gpt_keywords,id,n_comments,category,clean_text
41024,673ddd8496e992b8a87703a4,https://www.rtvslo.si/svet/vojna-v-ukrajini/ol...,[Igor E. Bergant],2024-11-20T11:35:22,"Olena Zelenska: Če se bomo nehali bojevati, bo...",Soproga ukrajinskega predsednika Volodimirja Z...,[{'caption': 'Olena Zelenska se je med obiskom...,"""Naša naloga je, da se naprej bojujemo za svoj...",[],svet,"[Humanitarna pomoč, Duševno zdravje, Vojaška a...",NaN,728051,205.0,NaN,soproga ukrajinskega predsednika volodimirja z...
52462,6816ae2957b40420fc1b4a42,https://www.rtvslo.si/zabava-in-slog/zanimivos...,[K. S.],2025-05-03T18:36:33,Za rojstni dan Donalda Trumpa se pripravlja ve...,"In medtem ko AP poroča, da parada še ni potrje...",[{'caption': 'Trump si že leta močno želi voja...,Ameriški mediji razkrivajo podrobne vojaške na...,"[Steve Warren, Dave Butler]",zabava-in-slog,"[Donald Trump, ZDA, Parada]",NaN,744560,NaN,NaN,ap poroča parada še potrjena fox news navaja i...
11365,6540f25e30638e06ced90e5e,https://www.rtvslo.si/kultura/drugo/vsak-dober...,"[Kaja Novosel, Program Ars]",2023-10-30T10:10:50,"""Vsak dober mornar mora preživeti eno nevihto""...",Teden na Uredništvu igranega programa zaznamuj...,[{'caption': 'Kratka radijska igra Na odprto j...,Čas neumorno in neslišno polzi mimo nas in sko...,"[Jože Rode, Aleš Valič, Jože Valentič, Darja D...",kultura,"[radijske igre, dan reformacije, dan spomina n...","[radijska igra, jesen, državni prazniki, nabor...",686605,0.0,NaN,teden uredništvu igranega programa zaznamuje m...


## Find regions for each data

In [38]:
#! FOUND EVEN FASTER WAY

# import re
# from collections import defaultdict

# # ─────────────────────────────────────────────────────────────────────────────
# # 1. Pre‑build mapping: term → set of region_id (once, before the loop)
# # ─────────────────────────────────────────────────────────────────────────────
# term_to_region = defaultdict(set)

# for _, row in df_naselja.iterrows():
#     rid = row['region_id']
#     for col in ('naselje', 'rodilnik', 'mestnik'):
#         term = str(row[col])
#         if term and term.lower() != 'nan':
#             term_to_region[term].add(rid)   # keep original case

# # ─────────────────────────────────────────────────────────────────────────────
# # 2. Build regex pattern from all terms (same as before)
# # ─────────────────────────────────────────────────────────────────────────────
# all_terms = set(term_to_region.keys())
# pattern = '|'.join(re.escape(term) for term in all_terms)
# regex = re.compile(rf'\b({pattern})\b', flags=re.IGNORECASE)

# # ─────────────────────────────────────────────────────────────────────────────
# # 3. Fast lookup function – no DataFrame scan per row
# # ─────────────────────────────────────────────────────────────────────────────
# def get_intersected_regions_fast(text_list):
#     full_text = " ".join(text_list)
#     found_words = set(regex.findall(full_text))

#     if not found_words:
#         return None

#     # Collect region IDs directly from the pre‑built mapping
#     region_ids = set()
#     for word in found_words:
#         # Keep original case for lookup (map is case‑sensitive)
#         # But regex.findall with IGNORECASE returns the word *as found*.
#         # If your terms have different case variants, unify them:
#         # word_lower = word.lower()
#         # for key in term_to_region: if key.lower() == word_lower ...
#         # Simpler: store mapping with lower‑case keys and lower‑case the found word.
#         #
#         # The clearest approach (avoids case mismatch):
#         # Build term_to_region with lower‑case keys, then use word.lower().
#         region_ids.update(term_to_region.get(word, set()))
#     return list(region_ids)

# # 4. Apply
# df['intersected_regions'] = df['paragraphs'].apply(get_intersected_regions_fast)

In [7]:
term_to_region = defaultdict(set)

for _, row in df_naselja.iterrows():
    rid = row['region_id']
    for col in ('naselje', 'rodilnik', 'mestnik'):
        term = str(row[col])
        if term and term.lower() != 'nan':
            term_to_region[term].add(rid)

In [8]:
import ahocorasick

# Build automaton with lowercase terms
automaton = ahocorasick.Automaton()
for term, region_ids in term_to_region.items():
    spaced_term1 = f" {term.lower()} " # Beseda more imeti space okrog sebe
    automaton.add_word(spaced_term1, (list(region_ids), term))  # Store just the IDs
automaton.make_automaton()

def get_regions_ahocorasick_fixed(text):
    region_ids = set()
    found_matches = {} 
    
    for _, (ids, word) in automaton.iter(text):
        region_ids.update(ids)
        found_matches[word] = ids

    #if found_matches:
        # Create a list of "Word (ID, ID)" strings
    #    display_list = [f"{word} {ids}" for word, ids in found_matches.items()]
    #    print(f"Matched: {', '.join(display_list)}")
    
    return list(region_ids) if region_ids else None
df['intersected_regions'] = df['clean_text'].apply(get_regions_ahocorasick_fixed)


In [9]:
novice_z_naselji_df = df[df['intersected_regions'].astype(bool)]

In [10]:
novice_z_naselji_df.count()

_id                    3559
url                    3559
authors                3504
date                   3559
title                  3559
paragraphs             3559
figures                3559
lead                   3555
mention                3559
topics                 3558
keywords               3559
gpt_keywords           1364
id                     3558
n_comments             2399
category                 14
clean_text             3559
intersected_regions    3559
dtype: int64

In [11]:
novice_z_naselji_df_za_db = novice_z_naselji_df[["id", "title", "url", "date", "topics", "paragraphs", "intersected_regions", "clean_text"]]

## Vectorize with TFIDF paragraphs

In [12]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vec = TfidfVectorizer(max_features=500, max_df=0.3, min_df=3) # Vektorji velikosti 500, max_df=0.3 => če se beseda pojavi 30%+ časa jo ignoriraj, min_df = minimalno kolikokrat se rabi beseda pojavit
tfidf_matrix = tfidf_vec.fit_transform(novice_z_naselji_df_za_db['clean_text'].fillna(''))
novice_z_naselji_df_za_db['tfidf'] = list(tfidf_matrix.toarray())

In [13]:
feature_names = tfidf_vec.get_feature_names_out()
np.save('final_data/tfidf_vocab.npy', feature_names, allow_pickle=True)

# Save to db

In [14]:
import sqlite3
print(sqlite3.sqlite_version)

3.45.3


In [15]:
connection = sqlite3.connect('final_data/novice.db') # Naredi db če še ne obstaja
connection.execute("PRAGMA foreign_keys = ON")
cursor = connection.cursor()

In [16]:
# Naredi db
cursor.execute("DELETE FROM novice")
cursor.execute("DELETE FROM regije")

cursor.execute('''
    CREATE TABLE IF NOT EXISTS regije (
        id char(5) PRIMARY KEY,
        name VARCHAR(50)
    )
''')

cursor.executemany("INSERT OR IGNORE INTO regije (id, name) VALUES (?, ?)", df_regije)

cursor.execute('''
    CREATE TABLE IF NOT EXISTS novice (
        id INTEGER PRIMARY KEY,
		title TEXT,
        url TEXT,
        date DATE,
        topic VARCHAR(30),
        content TEXT,
		clean_content TEXT,
		tfidf BLOB
    )
''')

cursor.execute('''
    CREATE TABLE IF NOT EXISTS novice_regije (
        novica_id INTEGER,
        regija_id char(5),
        PRIMARY KEY (novica_id, regija_id),
        FOREIGN KEY (novica_id) REFERENCES novice (id) ON DELETE CASCADE,
        FOREIGN KEY (regija_id) REFERENCES regije (id) ON DELETE CASCADE
    )
''')

cursor.execute("CREATE INDEX IF NOT EXISTS idx_novice_topic ON novice (topic)")
cursor.execute("CREATE INDEX IF NOT EXISTS idx_novice_date ON novice (date)")

connection.commit()
cursor.close()

In [17]:
def save_data_to_db(connection, df):
    cursor = connection.cursor()
    
    # 1. Prepare data for 'novice' table
    # We select only the columns that match the DB schema
    novice_items = df[['id', 'title', 'url', 'date','topics', 'paragraphs', 'clean_text', 'tfidf']].values.tolist()
    
    # 2. Prepare data for 'novice_regije' junction table
    # We iterate through the rows and 'explode' the intersected_regions list
    junction_items = []
    for _, row in df.iterrows():
        n_id = row['id']
        r_ids = row['intersected_regions'] # This is your list of IDs
        
        if isinstance(r_ids, list):
            for r_id in r_ids:
                junction_items.append((n_id, r_id))

    try:
        # Insert into main news table
        cursor.executemany('''
            INSERT OR REPLACE INTO novice (id, title, url, date, topic, content, clean_content, tfidf) 
            VALUES (?, ?, ?, ?, ?, ?, ?, ?)
        ''', novice_items)
        
        # Insert into junction table
        cursor.executemany('''
            INSERT OR IGNORE INTO novice_regije (novica_id, regija_id) 
            VALUES (?, ?)
        ''', junction_items)
        
        connection.commit()
        print(f"Successfully saved {len(df)} articles.")
        
    except Exception as e:
        connection.rollback()
        print(f"Error during save: {e}")
    finally:
        cursor.close()

#! UNCOMMENT IF NEED
save_data_to_db(connection, novice_z_naselji_df_za_db)

Successfully saved 3559 articles.


In [18]:
connection.close()